[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/06_alumno_no_supervisado.ipynb)

# MLY1101 · Machine Learning — Actividad 2.3
## Modelamiento no supervisado: encontrar estructura sin etiquetas

**Resultado de aprendizaje (RA2):** aplica modelos estadísticos al conjunto de datos
procesados para interpretarlos, utilizando metodologías ágiles, con la finalidad de obtener
conocimientos relevantes que permitan responder a las necesidades del contexto de negocio,
considerando aspectos éticos.

**Indicador de logro (IL 2.3):** elabora algoritmos de aprendizaje no supervisado para
descubrir patrones ocultos en los datos.

---

### La contracara de la Actividad 2.2

En la 2.2 había una etiqueta y se medía el acierto. Hoy **no hay etiqueta**, así que no se
puede acertar ni fallar. Y eso cambia todo:

| | Act. 2.2 · Supervisado | Act. 2.3 · No supervisado |
|---|---|---|
| Entrada | `X` e `y` | Solo `X` |
| Qué busca | Predecir `y` | Estructura oculta |
| Cómo se sabe si funcionó | Se mide contra `y` | **Hay que argumentarlo** |
| Riesgo típico | Fuga de información | **Encontrar patrones que no existen** |

Esa última fila es la sesión de hoy. K-medias **siempre** devuelve grupos: si le pides cuatro
sobre ruido puro, te da cuatro. Que existan no significa que signifiquen algo.

---

### La pregunta de hoy

> Sin decirle a nadie qué es cada objeto, ¿aparecen **grupos naturales** en las detecciones?
> ¿Y coinciden con los tipos que el sensor etiquetó?

---

### La idea central

> **Un grupo sin nombre no es un hallazgo.**

"Grupo 2" no le sirve a nadie. El trabajo empieza cuando el algoritmo termina: hay que mirar el
perfil de cada grupo y poder decir *"objetos grandes y rápidos"*, *"objetos cercanos al sensor"*.
Eso no lo hace el algoritmo. Lo haces tú, y es lo que se evalúa.

---

### Al final de la sesión debes entregar

Un **informe de segmentación** con:

- la justificación del número de grupos elegido, con **dos** criterios y qué haces si discrepan;
- el perfil e interpretación de cada grupo, con nombre propio;
- el contraste con la etiqueta conocida, y qué te dice sobre lo que descubriste;
- una conclusión honesta sobre si la estructura encontrada es útil para el negocio.

---
## Preparación del entorno

Como en la Actividad 2.2, no reescribimos nada: importamos los mismos nodos del pipeline.

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "kedro_mly1101" / "src"))

RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"
RUTA_PARAMETROS = RAIZ / "kedro_mly1101" / "conf" / "base" / "parameters.yml"

print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from kedro_mly1101.pipelines.preprocesamiento import nodes as limpieza
from kedro_mly1101.pipelines.no_supervisado import nodes as grupos

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

PARAMETROS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))
CONFIG = PARAMETROS["agrupamiento"]

crudo = pd.read_csv(RUTA_DATOS)
paso = limpieza.normalizar_categorias(crudo, PARAMETROS["mapas_categorias"])
paso = limpieza.descubrir_faltantes(paso, PARAMETROS["centinelas"])
paso = limpieza.marcar_imposibles(paso, PARAMETROS["reglas_dominio"])
limpio = limpieza.quitar_duplicados_y_constantes(paso, PARAMETROS["columnas_a_descartar"])

print(f"Datos limpios: {limpio.shape[0]:,} filas")
print("Variables para agrupar:", CONFIG["variables"])
print("Etiqueta de contraste (NO se usa para agrupar):", CONFIG["etiqueta_de_contraste"])

---
# Bloque 1 · ⭐ Escalar no es opcional

K-medias agrupa por **distancia euclídea**. Y la distancia suma los cuadrados de las diferencias
de cada variable, **sin importar en qué unidad estén**.

Mira nuestras variables:

| Variable | Rango típico |
|---|---|
| `box_height` | ~0,5 a 4 metros |
| `num_lidar_points` | 0 a **varios miles** |

Una diferencia de 500 puntos láser pesa cientos de veces más que una diferencia de 2 metros de
altura. Sin escalar, el "agrupamiento por objeto" sería en realidad **un agrupamiento por número
de puntos** con otro nombre.

### ✏️ TODO 1 — Ver el problema antes de arreglarlo

Compara la desviación típica de cada variable **antes** de escalar.

In [ ]:
# TODO 1: ¿qué tan distintas son las escalas de las variables?
antes = limpio[CONFIG["variables"]].____().T[["mean", "std", "min", "max"]]
print(antes.round(2).to_string())
print(f"\nRazón entre la mayor y la menor desviación típica: "
      f"{antes['std'].____() / antes['std'].min():,.0f}×")

### ✏️ TODO 2 — Escalar

`grupos.preparar_matriz()` selecciona las variables, descarta filas incompletas y las
estandariza: media 0 y desviación típica 1.

In [ ]:
# TODO 2: escala las variables.
matriz = grupos.____(limpio, CONFIG)

despues = matriz[CONFIG["variables"]].describe().T[["mean", "std"]]
print(despues.round(4).to_string())
print(f"\nFilas: {len(matriz):,}")

In [ ]:
# Autochequeo
assert np.allclose(matriz[CONFIG["variables"]].mean(), 0, atol=1e-9), "las medias deben ser 0"
assert np.allclose(matriz[CONFIG["variables"]].std(ddof=0), 1, atol=1e-9), "las desviaciones, 1"
assert CONFIG["etiqueta_de_contraste"] in matriz.columns, (
    "la etiqueta debe viajar en la tabla, aunque no se use para agrupar"
)
print("✅ Todas las variables en la misma escala. Ahora una diferencia de 1 significa")
print("   lo mismo en cualquiera de ellas: una desviación típica.")

---
# Bloque 2 · ⭐⭐ ¿Cuántos grupos?

K-medias necesita que le digas `k` de antemano. Y no hay una respuesta correcta: hay dos
criterios que a veces se contradicen.

| Criterio | Qué mide | Su trampa |
|---|---|---|
| **Inercia** | Suma de distancias al centro de su grupo | **Siempre baja** al añadir grupos |
| **Silueta** | Qué tan separados están los grupos entre sí | Puede no tener máximo claro |

La inercia por sí sola no decide nada: si le hicieras caso, terminarías con un grupo por fila e
inercia cero.

### ✏️ TODO 3 — Probar varios `k`

In [ ]:
# TODO 3: prueba varios números de grupos y mide los dos criterios.
busqueda = grupos.____(matriz, CONFIG)
print(busqueda.to_string(index=False))

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
ejes[0].plot(busqueda["k"], busqueda["____"], marker="o")
ejes[0].set_title("Inercia (siempre baja)")
ejes[0].set_xlabel("k")
ejes[1].plot(busqueda["k"], busqueda["____"], marker="o", color="darkorange")
ejes[1].set_title("Silueta (más alto es mejor)")
ejes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

In [ ]:
# Autochequeo
assert busqueda["inercia"].is_monotonic_decreasing, (
    "la inercia SIEMPRE baja al añadir grupos: por eso sola no sirve"
)
mejor_k = int(busqueda.loc[busqueda["silueta"].idxmax(), "k"])
print(f"✅ La silueta tiene su máximo en k = {mejor_k}.")
print(f"   Pero el pipeline usa k = {CONFIG['k']}. Esa discrepancia es el ejercicio.")

### ✏️ TODO 4 — La discrepancia

La silueta prefiere un número de grupos y el pipeline usa otro. **No es un error:** es la
situación normal.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. ¿Qué `k` elegirías tú **solo con estos dos gráficos**?
2. ¿Qué información te haría cambiar de opinión?

---
# Bloque 3 · ⭐⭐ Agrupar, y después el trabajo de verdad

Ejecutar K-medias es una línea. Lo que sigue es la actividad.

### ✏️ TODO 5 — Agrupar

In [ ]:
# TODO 5: agrupa con el k configurado.
agrupada = grupos.____(matriz, CONFIG)

print(f"k = {CONFIG['k']}")
print(agrupada["grupo"].value_counts().sort_index().to_string())

### ✏️ TODO 6 — Perfilar: ponerle nombre a cada grupo

Aquí está el trabajo. La tabla siguiente da la media de cada variable **en unidades de
desviación típica**: un `+2` significa "dos desviaciones por encima de la media general".

In [ ]:
# TODO 6: describe cada grupo por la media de sus variables.
perfil = grupos.____(agrupada, CONFIG)
perfil

### ✏️ TODO 7 — Los nombres

Mira el perfil y bautiza cada grupo. Un nombre que un colega entienda sin ver la tabla.

*Pista: fíjate especialmente en el grupo más pequeño. Sus valores no se parecen a nada.*

In [ ]:
# TODO 7: ponle nombre a cada grupo, mirando su perfil.
nombres = {
    0: "____",
    1: "____",
    2: "____",
    3: "____",
}
for numero, nombre in nombres.items():
    fila = perfil[perfil["grupo"] == numero].iloc[0]
    print(f"Grupo {numero}: {nombre:52s} {fila['n']:6,.0f} filas ({fila['pct']:5.2f} %)")

In [ ]:
# Autochequeo
assert all(n and not n.startswith("____") for n in nombres.values()), "faltan nombres"
mas_pequeno = perfil.loc[perfil["n"].idxmin()]
print(f"✅ El grupo más pequeño es el {int(mas_pequeno['grupo'])}: "
      f"{mas_pequeno['n']:,.0f} filas ({mas_pequeno['pct']:.2f} %)")
print(f"   box_length está a {mas_pequeno['box_length']:+.2f} desviaciones típicas.")
print("   ¿Qué objeto del tránsito es enorme, poco frecuente, y ya apareció en la EA1?")

---
# Bloque 4 · Contrastar con la etiqueta: **no es una evaluación**

Tenemos `object_type` guardado y sin usar. Ahora lo sacamos.

**Cuidado con lo que esto es y lo que no es.** El algoritmo nunca vio la etiqueta, así que no
puede acertar ni fallar. Esto es una **comprobación de sentido**: ¿la estructura que apareció
sola tiene alguna lectura conocida?

- Si un grupo concentra un tipo de objeto, encontraste algo con significado de dominio.
- Si todos los grupos tienen la misma mezcla, no encontraste nada útil, **por buena que sea la
  silueta**.

### ✏️ TODO 8 — El cruce

In [ ]:
# TODO 8: cruza los grupos descubiertos con la etiqueta conocida.
cruce = grupos.____(agrupada, CONFIG)
print(cruce.to_string(index=False), "\n")

sns.heatmap(cruce.set_index("grupo"), annot=True, fmt=".1f", cmap="Blues", cbar=False)
plt.title("% de cada tipo de objeto dentro de cada grupo")
plt.tight_layout()
plt.show()

### ✏️ TODO 9 — Leerlo bien

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. ¿Los grupos coinciden con los tipos de objeto? Responde con cifras.
2. Hay dos grupos que son casi 100 % `vehicle`. ¿Es un error del algoritmo? ¿Qué los distingue?
3. Un grupo mezcla peatones y señalética. ¿Por qué el algoritmo los junta, si para el negocio no
   tienen nada que ver?

---
# Bloque 5 · Reducción de dimensionalidad

Cinco variables no se pueden dibujar. **PCA** busca las direcciones de máxima varianza y
proyecta los datos sobre las primeras, conservando toda la información que pueda.

La cifra que importa no es cuánto explica la primera componente, sino **cuántas necesitas para
conservar el 90 %**. Si con pocas basta, tus variables eran redundantes entre sí — y eso también
es un hallazgo.

### ✏️ TODO 10 — ¿Cuánta redundancia hay?

In [ ]:
# TODO 10: ¿cuánta información conserva cada componente?
varianza = grupos.____(matriz, CONFIG)
print(varianza.to_string(index=False))

n90 = int((varianza["varianza_acumulada"] < 0.90).sum() + 1)
print(f"\nCon {n90} de {len(varianza)} componentes se conserva el 90 % de la varianza.")

### ✏️ TODO 11 — Dibujar, sabiendo lo que se pierde

In [ ]:
# TODO 11: proyecta a dos dimensiones y dibuja.
proyeccion = grupos.____(matriz, CONFIG)
proyeccion["grupo"] = agrupada["grupo"].to_numpy()

explicada = varianza.loc[1, "varianza_acumulada"]

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
muestra = proyeccion.sample(4000, random_state=42)
sns.scatterplot(data=muestra, x="componente_1", y="componente_2", hue="____",
                palette="tab10", s=8, ax=ejes[0], legend="full")
ejes[0].set_title("Coloreado por GRUPO descubierto")
sns.scatterplot(data=muestra, x="componente_1", y="componente_2",
                hue=CONFIG["etiqueta_de_contraste"], palette="Set2", s=8, ax=ejes[1])
ejes[1].set_title("Coloreado por tipo de objeto REAL")
plt.suptitle(f"Proyección 2D — conserva el {100*explicada:.1f} % de la varianza")
plt.tight_layout()
plt.show()

In [ ]:
# Autochequeo
assert abs(varianza["varianza_acumulada"].iloc[-1] - 1.0) < 1e-6, (
    "todas las componentes juntas deben explicar el 100 %"
)
print(f"✅ La proyección 2D conserva el {100*explicada:.1f} % de la varianza.")
print(f"   Es decir: el {100*(1-explicada):.1f} % de la información NO está en ese dibujo.")
print("   Dos puntos que se ven pegados pueden estar lejos en el espacio original.")

---
# Cierre · Informe de segmentación

Esta es la entrega de la Actividad 2.3. Máximo una página.

---

### El problema

**Qué buscábamos sin etiquetas:** `____`
**Variables usadas para agrupar:** `____`
**Por qué escalé antes de agrupar:** `____`

### Cuántos grupos, y por qué

| k | Inercia | Silueta |
|---|---|---|
| | | |

**k elegido:** `____`
**Criterio estadístico que lo respalda (o no):** `____`
**Argumento de dominio:** `____`

> Si tu k no es el de la silueta máxima, **eso está bien** y hay que defenderlo. Si sí lo es,
> también — pero di por qué el contenido de los grupos lo confirma.

### Los grupos, con nombre

| Grupo | Nombre | % de las filas | Qué lo caracteriza |
|---|---|---|---|
| 0 | `____` | | |
| 1 | `____` | | |
| 2 | `____` | | |
| 3 | `____` | | |

### Contraste con la etiqueta conocida

**¿Los grupos coinciden con los tipos de objeto?** `____`
**Un grupo que NO corresponde a un tipo, y qué significa:** `____`
**Dos tipos que el algoritmo confunde, y por qué:** `____`
**Qué variable habría que agregar para separarlos:** `____`

### Reducción de dimensionalidad

**Componentes para conservar el 90 %:** `____` de `____`
**Qué dice eso sobre mis variables:** `____`
**Varianza que conserva mi gráfico 2D:** `____` %

### La conclusión honesta

**¿Esta segmentación le sirve a alguien?** `____`

*(Responder "no del todo" está permitido y a veces es lo correcto. Lo que se evalúa es el
argumento: qué decisión podría tomar alguien con estos grupos que no pudiera tomar sin ellos.)*

**Qué haría distinto con más tiempo:** `____`